Hope this gets smoothly . 

In [11]:
import ast  # Text dictionary ko asli Python dictionary banane ke liye
import numpy as np  # Numbers aur arrays handle karne ke liye
import pandas as pd  # CSV tables padhne ke liye
import wfdb  # ECG waveform (.dat/.hea) padhne ke liye

# Apne PTB-XL folder ka rasta
DATA_PATH = '../PTBXL dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'

print('Step 1 Complete: Libraries aur Path set ho gaye!')


Step 1 Complete: Libraries aur Path set ho gaye!


In [12]:
# Master spreadsheet load karo
df = pd.read_csv(DATA_PATH + 'ptbxl_database.csv', index_col='ecg_id')

print('Total ECGs:', len(df))
print('Columns ke naam:', list(df.columns))


Total ECGs: 21837
Columns ke naam: ['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


In [13]:
df.info()  # DataFrame ke baare mein basic info dikhata hai

<class 'pandas.DataFrame'>
RangeIndex: 21837 entries, 1 to 21837
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   patient_id                    21837 non-null  float64
 1   age                           21748 non-null  float64
 2   sex                           21837 non-null  int64  
 3   height                        6983 non-null   float64
 4   weight                        9429 non-null   float64
 5   nurse                         20328 non-null  float64
 6   site                          21819 non-null  float64
 7   device                        21837 non-null  str    
 8   recording_date                21837 non-null  str    
 9   report                        21837 non-null  str    
 10  scp_codes                     21837 non-null  str    
 11  heart_axis                    13332 non-null  str    
 12  infarction_stadium1           5626 non-null   str    
 13  infarction_s

In [14]:
# Includes the first occurrence and all repeats
duplicates = df[df.duplicated(subset=['patient_id'], keep=False)]
duplicates

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
12,11031.0,43.0,1,NaN,44.0,2.0,0.0,CS-12 E,1984-12-21 08:02:26,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,8,records100/00000/00012_lr,records500/00000/00012_hr
17,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-01-20 14:25:25,vorhof:hf 280 tachykardes vorhofflimmern uebe...,...,True,NaN,", alles,",NaN,NaN,NaN,NaN,9,records100/00000/00017_lr,records500/00000/00017_hr
18,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-01-20 15:17:24,"vorhofflattern 2:1 Überleitung,jetzt nach isop...",...,True,NaN,", alles,",NaN,NaN,NaN,NaN,9,records100/00000/00018_lr,records500/00000/00018_hr
20,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-01-23 12:55:32,supraventrikulÄre ersatzsystole(n) interponier...,...,True,NaN,NaN,NaN,NaN,VES,NaN,9,records100/00000/00020_lr,records500/00000/00020_hr
23,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-01-25 12:49:40,"2:1 Überleitung bei vorhofflattern,jetzt unter...",...,True,NaN,NaN,NaN,NaN,NaN,NaN,9,records100/00000/00023_lr,records500/00000/00023_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21803,12842.0,87.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-02-07 21:13:49,sinus arrhythmie schwierig bestimmbare qrs-ach...,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,2,records100/21000/21803_lr,records500/21000/21803_hr
21804,12842.0,87.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-02-11 08:55:05,tachykardes vorhofflimmern lagetyp normal st &...,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/21000/21804_lr,records500/21000/21804_hr
21823,18689.0,80.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-06 18:57:06,"sinusrhythmus linkstyp st & t abnorm, wahrsche...",...,True,NaN,NaN,NaN,NaN,NaN,NaN,3,records100/21000/21823_lr,records500/21000/21823_hr


In [15]:
import pandas as pd

cols = [
    'sex',
    'scp_codes',
    'heart_axis',
    'infarction_stadium1',
    'infarction_stadium2',
    'validated_by',
    'second_opinion',
    'initial_autogenerated_report',
    'validated_by_human',
    'baseline_drift',
    'static_noise',
    'burst_noise',
    'electrodes_problems',
    'extra_beats',
    'pacemaker',
    'strat_fold',
]

unique_per_col = {}
for col in cols:
  if col == 'scp_codes' and df[col].apply(lambda x: isinstance(x, dict)).any():
    # Extract all unique diagnostic keys across every dictionary in scp_codes
    unique_per_col[col] = sorted(
        {code for d in df[col].dropna() for code in d.keys()}
    )
  else:
    unique_per_col[col] = df[col].dropna().unique().tolist()

for col, vals in unique_per_col.items():
  print(f'{col} ({len(vals)}): {vals}\n')


sex (2): [1, 0]

scp_codes (5466): ["{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}", "{'NORM': 80.0, 'SBRAD': 0.0}", "{'NORM': 100.0, 'SR': 0.0}", "{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}", "{'NORM': 80.0, 'SARRH': 0.0}", "{'NORM': 100.0, 'SARRH': 0.0}", "{'AFLT': 100.0, 'ABQRS': 0.0, 'AFIB': 0.0}", "{'AFLT': 100.0}", "{'AFLT': 100.0, 'ABQRS': 0.0}", "{'NDT': 100.0, 'SR': 0.0}", "{'NST_': 100.0, 'DIG': 100.0, 'SR': 0.0}", "{'NORM': 80.0, 'SR': 0.0}", "{'LVH': 100.0, 'SR': 0.0}", "{'LPFB': 100.0, 'SR': 0.0}", "{'IMI': 15.0, 'LNGQT': 100.0, 'NST_': 100.0, 'DIG': 100.0, 'ABQRS': 0.0, 'SR': 0.0}", "{'LAFB': 100.0, 'SR': 0.0}", "{'IRBBB': 100.0, 'RAO/RAE': 50.0, 'RVH': 35.0, 'SR': 0.0}", "{'IVCD': 100.0, 'SR': 0.0}", "{'LMI': 15.0, 'IVCD': 100.0, 'SR': 0.0}", "{'IRBBB': 100.0, 'SR': 0.0}", "{'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}", "{'ASMI': 15.0, 'ABQRS': 0.0, 'SR': 0.0}", "{'NORM': 100.0, 'SBRAD': 0.0}", "{'AMI': 50.0, 'IRBBB': 100.0, 'SR': 0.0}", "{'NDT': 100.0, 'IRBBB': 100.0, 'SR': 0.0}", "

In [16]:
# 1. 'scp_codes' abhi plain text (string) hai, use asli Python dictionary banao
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# 2. Medical dictionary (scp_statements.csv) load karo
agg_df = pd.read_csv(DATA_PATH + 'scp_statements.csv', index_col=0)

# 3. Sirf wahi rows rakho jo bimari (diagnostic == 1) batati hain (total 44 codes)
agg_df = agg_df[agg_df.diagnostic == 1]


# 4. Ek chhota function jo har ECG ke codes padh kar unki superclass nikaalega
def aggregate_diagnostic(y_dic):
  tmp = []
  for key in y_dic.keys():  # ECG ke har code ko check karo
    if key in agg_df.index:  # Kya ye code un 44 diagnostic codes mein hai?
      tmp.append(
          agg_df.loc[key].diagnostic_class
      )  # Uski superclass list mein daal do
  return list(set(tmp))  # Duplicate hata kar unique list lauta do


# 5. Ye function poore dataset par lagao aur naya column 'diagnostic_superclass' banao
df['diagnostic_superclass'] = df.scp_codes.apply(aggregate_diagnostic)

# 6. Dekho kaisa result aaya!
print(df[['scp_codes', 'diagnostic_superclass']].head(10))


                                       scp_codes diagnostic_superclass
ecg_id                                                                
1       {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}                [NORM]
2                   {'NORM': 80.0, 'SBRAD': 0.0}                [NORM]
3                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
4                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
5                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
6                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
7                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
8         {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}                  [MI]
9                     {'NORM': 100.0, 'SR': 0.0}                [NORM]
10                    {'NORM': 100.0, 'SR': 0.0}                [NORM]


In [17]:
# 1. Check karo abhi total kitni rows hain
print('Filtering se pehle total rows:', len(df))

# 2. Sirf wahi rows rakho jinmein kam se kam 1 diagnostic superclass ho (khali [] hatao)
df = df[df['diagnostic_superclass'].map(len) > 0]
print('Filtering ke baad clean rows:', len(df))

# 3. Dekho poore dataset mein har bimari ke kitne patients/ECGs hain!
print('\nHar Superclass ki ginti:')
print(df['diagnostic_superclass'].explode().value_counts())


Filtering se pehle total rows: 21837
Filtering ke baad clean rows: 21430

Har Superclass ki ginti:
diagnostic_superclass
NORM    9528
MI      5486
STTC    5250
CD      4907
HYP     2655
Name: count, dtype: int64


In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Multi-label encoder banayo
mlb = MultiLabelBinarizer()

# 2. 'diagnostic_superclass' wali lists ko 0 aur 1 ke columns mein badlo
Y_labels = mlb.fit_transform(df['diagnostic_superclass'])

# 3. Dekho kaunsa column kis bimari ka hai aur pehle 10 patients ke labels kaise dikhte hain!
print('5 Columns ka order:', mlb.classes_)
print('Y_labels ka shape:', Y_labels.shape)  # (21430, 5) hona chahiye
print('\nPehle 10 patients ke multi-label vectors:\n', Y_labels[:10])


5 Columns ka order: ['CD' 'HYP' 'MI' 'NORM' 'STTC']
Y_labels ka shape: (21430, 5)

Pehle 10 patients ke multi-label vectors:
 [[0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 1 0]]


In [19]:
# 1. 'strat_fold' column ke hisaab se True/False masks banao
train_mask = (df['strat_fold'] <= 8).values
val_mask = (df['strat_fold'] == 9).values
test_mask = (df['strat_fold'] == 10).values

# 2. Labels (Y) ko teeno hisson mein baanto
y_train = Y_labels[train_mask]
y_val = Y_labels[val_mask]
y_test = Y_labels[test_mask]

# 3. File paths (100 Hz) ko bhi teeno hisson mein baanto
train_files = df.loc[train_mask, 'filename_lr'].values
val_files = df.loc[val_mask, 'filename_lr'].values
test_files = df.loc[test_mask, 'filename_lr'].values

# 4. Ginti check karo!
print('Training ECGs (Folds 1-8):', len(y_train))
print('Validation ECGs (Fold 9) :', len(y_val))
print('Test ECGs (Fold 10)      :', len(y_test))


Training ECGs (Folds 1-8): 17111
Validation ECGs (Fold 9) : 2156
Test ECGs (Fold 10)      : 2163


In [20]:
import os


# 1. Ek chhota function jo file paths ki list lega aur saare signals load karega
def load_signals(file_list, base_path):
  signals = []
  for f in file_list:
    # wfdb.rdsamp() se sirf signal array ([0]) nikaalo, metadata ([1]) chhod do
    sig, _ = wfdb.rdsamp(base_path + f)
    signals.append(sig)
  return np.array(signals, dtype=np.float32)


print('Training signals load ho rahe hain... (thoda intezar karein)')
X_train = load_signals(train_files, DATA_PATH)

print('Validation signals load ho rahe hain...')
X_val = load_signals(val_files, DATA_PATH)

print('Test signals load ho rahe hain...')
X_test = load_signals(test_files, DATA_PATH)

# Shapes check karo!
print('\nLoading Complete!')
print('X_train shape:', X_train.shape)  # (~17100, 1000, 12)
print('X_val shape  :', X_val.shape)  # (~2150, 1000, 12)
print('X_test shape :', X_test.shape)  # (~2150, 1000, 12)


Training signals load ho rahe hain... (thoda intezar karein)
Validation signals load ho rahe hain...
Test signals load ho rahe hain...

Loading Complete!
X_train shape: (17111, 1000, 12)
X_val shape  : (2156, 1000, 12)
X_test shape : (2163, 1000, 12)


In [21]:
os.makedirs('processed_data', exist_ok=True)

np.save('processed_data/X_train.npy', X_train)
np.save('processed_data/y_train.npy', y_train)

np.save('processed_data/X_val.npy', X_val)
np.save('processed_data/y_val.npy', y_val)

np.save('processed_data/X_test.npy', X_test)
np.save('processed_data/y_test.npy', y_test)

print('Saara data processed_data/ folder mein safely save ho gaya!')


Saara data processed_data/ folder mein safely save ho gaya!


In [22]:
import numpy as np

# 1. Saved .npy files load karo (0.5 second mein load hoga!)
X_train = np.load('processed_data/X_train.npy')
y_train = np.load('processed_data/y_train.npy')

X_val = np.load('processed_data/X_val.npy')
y_val = np.load('processed_data/y_val.npy')

X_test = np.load('processed_data/X_test.npy')
y_test = np.load('processed_data/y_test.npy')

# 2. Sirf X_train se har lead ka Mean aur Std nikaalo
# axis=(0, 1) ka matlab: saare patients (0) aur saare time-steps (1) par average lo, har lead (2) ke liye alag
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8  # 1e-8 taaki zero se divide na ho

# 3. Teeno sets ko normalize karo
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

# 4. PyTorch ke liye shape ko (Samples, 1000, 12) se (Samples, 12, 1000) banao
X_train = np.transpose(X_train, (0, 2, 1))
X_val = np.transpose(X_val, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

print('Ready for PyTorch!')
print('X_train shape:', X_train.shape)  # (~17100, 12, 1000)
print('X_val shape  :', X_val.shape)
print('X_test shape :', X_test.shape)


Ready for PyTorch!
X_train shape: (17111, 12, 1000)
X_val shape  : (2156, 12, 1000)
X_test shape : (2163, 12, 1000)


In [23]:
import numpy as np
import torch

# 1. GPU Check
print(
    '1. GPU:',
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only',
)

# 2. NaN / Inf Check
print(
    '2. Any NaN in X_train?',
    np.isnan(X_train).any(),
    '| Any Inf?',
    np.isinf(X_train).any(),
)

# 3. Normalization Check (Mean ~ 0, Std ~ 1 hona chahiye)
print(
    f'3. X_train Mean: {X_train.mean():.4f} | Std: {X_train.std():.4f}'
)

# 4. Labels Check (Har class ka % teeno splits mein lagbhag barabar hona chahiye)
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
print('4. Class % (Train | Val | Test):')
for i, c in enumerate(classes):
  tr_p = y_train[:, i].mean() * 100
  va_p = y_val[:, i].mean() * 100
  te_p = y_test[:, i].mean() * 100
  print(f'   {c:4s}: {tr_p:5.1f}% | {va_p:5.1f}% | {te_p:5.1f}%')


1. GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
2. Any NaN in X_train? False | Any Inf? False
3. X_train Mean: 0.0000 | Std: 1.0216
4. Class % (Train | Val | Test):
   CD  :  22.9% |  23.1% |  23.0%
   HYP :  12.4% |  12.6% |  12.2%
   MI  :  25.7% |  25.2% |  25.6%
   NORM:  44.5% |  44.4% |  44.6%
   STTC:  24.5% |  24.8% |  24.2%


In [24]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 1. GPU device set karo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# 2. NumPy arrays ko PyTorch Tensors mein badlo
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
)

# 3. DataLoaders banao (64 ECGs ek baar mein GPU par jayenge)
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Ek batch nikaal kar verify karo
sample_x, sample_y = next(iter(train_loader))
print('Batch X shape:', sample_x.shape)  # (64, 12, 1000) hona chahiye
print('Batch y shape:', sample_y.shape)  # (64, 5) hona chahiye


Using device: cuda
Batch X shape: torch.Size([64, 12, 1000])
Batch y shape: torch.Size([64, 5])


In [25]:
class SimpleECGCNN(nn.Module):

  def __init__(self, num_classes=5):
    super().__init__()

    # Block 1: 12 leads -> 32 feature maps
    self.conv1 = nn.Sequential(
        nn.Conv1d(
            in_channels=12, out_channels=32, kernel_size=7, stride=2, padding=3
        ),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.MaxPool1d(kernel_size=2),
    )

    # Block 2: 32 -> 64 feature maps
    self.conv2 = nn.Sequential(
        nn.Conv1d(
            in_channels=32, out_channels=64, kernel_size=5, stride=2, padding=2
        ),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.MaxPool1d(kernel_size=2),
    )

    # Block 3: 64 -> 128 feature maps
    self.conv3 = nn.Sequential(
        nn.Conv1d(
            in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1
        ),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.AdaptiveAvgPool1d(1),  # Time dimension ko 1 kar deta hai
    )

    # Final Classifier: 128 features -> 5 diseases
    self.fc = nn.Linear(128, num_classes)

  def forward(self, x):
    x = self.conv1(x)
    x = self.conv2(x)
    x = self.conv3(x)
    x = x.squeeze(-1)  # Shape (64, 128, 1) se (64, 128) banao
    x = self.fc(x)  # Shape (64, 5)
    return x


# Model ko RTX 3050 GPU par bhejo
model = SimpleECGCNN(num_classes=5).to(device)

# Ek batch daal kar test karo ki model bina error ke output deta hai ya nahi
dummy_out = model(sample_x.to(device))
print('Model Output Shape:', dummy_out.shape)  # (64, 5) aana chahiye!
print('Total Parameters  :', sum(p.numel() for p in model.parameters()))


Model Output Shape: torch.Size([64, 5])
Total Parameters  : 38821


In [27]:
from sklearn.metrics import roc_auc_score

WINDOW, STRIDE = 250, 125   # 2.5s @ 100 Hz, half-overlap


def predict_tta(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            win = [torch.sigmoid(model(batch_x[:, :, s:s + WINDOW]))
                   for s in range(0, batch_x.shape[2] - WINDOW + 1, STRIDE)]
            preds.append(torch.stack(win).max(dim=0).values.cpu().numpy())
            targets.append(batch_y.numpy())
    return np.vstack(preds), np.vstack(targets)


# 1. Loss Function aur Optimizer set karo
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 15
best_val_auc = 0.0

for epoch in range(1, EPOCHS + 1):
  # --- TRAINING PHASE ---
  model.train()
  train_loss = 0.0

  for batch_x, batch_y in train_loader:
    # har batch se ek random 2.5s window kaato
    start = np.random.randint(0, batch_x.shape[2] - WINDOW + 1)
    batch_x = batch_x[:, :, start:start + WINDOW]
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()
    outputs = model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()

  # --- VALIDATION PHASE (Fold 9, TTA ke saath) ---
  val_preds, val_targets = predict_tta(val_loader)
  val_auc = roc_auc_score(val_targets, val_preds, average='macro')

  # Sabse achhe validation score wale model ko save kar lo
  if val_auc > best_val_auc:
    best_val_auc = val_auc
    torch.save(model.state_dict(), 'best_cnn_tta.pth')

  print(
      f'Epoch {epoch:02d}/{EPOCHS} | Train Loss:'
      f' {train_loss/len(train_loader):.4f} | Val Macro-AUC: {val_auc:.4f}'
  )

print(f'\nTraining Complete! Best Validation Macro-AUC: {best_val_auc:.4f}')

Epoch 01/15 | Train Loss: 0.3208 | Val Macro-AUC: 0.8990
Epoch 02/15 | Train Loss: 0.3061 | Val Macro-AUC: 0.9066
Epoch 03/15 | Train Loss: 0.2961 | Val Macro-AUC: 0.9107
Epoch 04/15 | Train Loss: 0.2931 | Val Macro-AUC: 0.9117
Epoch 05/15 | Train Loss: 0.2876 | Val Macro-AUC: 0.9129
Epoch 06/15 | Train Loss: 0.2848 | Val Macro-AUC: 0.9126
Epoch 07/15 | Train Loss: 0.2798 | Val Macro-AUC: 0.9168
Epoch 08/15 | Train Loss: 0.2769 | Val Macro-AUC: 0.9180
Epoch 09/15 | Train Loss: 0.2743 | Val Macro-AUC: 0.9189
Epoch 10/15 | Train Loss: 0.2717 | Val Macro-AUC: 0.9197
Epoch 11/15 | Train Loss: 0.2723 | Val Macro-AUC: 0.9204
Epoch 12/15 | Train Loss: 0.2690 | Val Macro-AUC: 0.9190
Epoch 13/15 | Train Loss: 0.2653 | Val Macro-AUC: 0.9249
Epoch 14/15 | Train Loss: 0.2658 | Val Macro-AUC: 0.9229
Epoch 15/15 | Train Loss: 0.2628 | Val Macro-AUC: 0.9227

Training Complete! Best Validation Macro-AUC: 0.9249


In [28]:
from sklearn.metrics import f1_score, roc_auc_score
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']

model.load_state_dict(torch.load('best_cnn_tta.pth'))
model.eval()

# 1. Fold 9 par TTA se predictions -> INHI par threshold tune karo
val_preds, val_targets = predict_tta(val_loader)

grid = np.linspace(0.05, 0.95, 91)
thresholds = np.array([
    grid[int(np.argmax([f1_score(val_targets[:, i], (val_preds[:, i] >= t).astype(int))
                        for t in grid]))]
    for i in range(5)
])
print('Tuned thresholds:', dict(zip(classes, thresholds.round(2))))

# 2. Fold 10 par TTA se predictions
test_preds, test_targets = predict_tta(test_loader)

# 3. Metrics
macro_auc = roc_auc_score(test_targets, test_preds, average='macro')
binary_preds = (test_preds >= thresholds).astype(int)
macro_f1 = f1_score(test_targets, binary_preds, average='macro')

print('==========================================')
print(f'FINAL TEST MACRO-AUC (Fold 10): {macro_auc:.4f}')
print(f'FINAL TEST MACRO-F1  (Fold 10): {macro_f1:.4f}')
print('==========================================')

print('\nPer-Class Breakdown:')
for i, c in enumerate(classes):
  print(f'  {c:4s} -> AUC: {roc_auc_score(test_targets[:, i], test_preds[:, i]):.4f}'
        f' | F1: {f1_score(test_targets[:, i], binary_preds[:, i]):.4f}')

Tuned thresholds: {'CD': np.float64(0.58), 'HYP': np.float64(0.34), 'MI': np.float64(0.57), 'NORM': np.float64(0.55), 'STTC': np.float64(0.52)}
FINAL TEST MACRO-AUC (Fold 10): 0.9157
FINAL TEST MACRO-F1  (Fold 10): 0.7329

Per-Class Breakdown:
  CD   -> AUC: 0.9117 | F1: 0.7380
  HYP  -> AUC: 0.8959 | F1: 0.6171
  MI   -> AUC: 0.9100 | F1: 0.7057
  NORM -> AUC: 0.9337 | F1: 0.8482
  STTC -> AUC: 0.9274 | F1: 0.7554


In [29]:
rng = np.random.default_rng(42)
n = len(test_targets)
aucs, f1s = [], []

while len(aucs) < 1000:
    idx = rng.integers(0, n, n)
    yt, yp = test_targets[idx], test_preds[idx]
    # agar kisi class ke saare positive ya saare negative gayab -> dobara draw karo
    if (yt.sum(axis=0) == 0).any() or (yt.sum(axis=0) == n).any():
        continue
    aucs.append(roc_auc_score(yt, yp, average='macro'))
    f1s.append(f1_score(yt, (yp >= thresholds).astype(int), average='macro'))

for name, arr, point in [('Macro-AUC', aucs, macro_auc), ('Macro-F1', f1s, macro_f1)]:
    lo, hi = np.percentile(arr, [2.5, 97.5])
    print(f'{name}: {point:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')

Macro-AUC: 0.9157  95% CI [0.9080, 0.9227]
Macro-F1: 0.7329  95% CI [0.7174, 0.7478]


In [33]:
import time

# 1. Test set (Fold 10) mein se koi ek patient uthao (maan lo index 5)
patient_idx = 5
single_ecg = torch.tensor(
    X_test[patient_idx : patient_idx + 1], dtype=torch.float32
).to(device)
true_label = y_test[patient_idx]

# 2. Model se prediction lo aur exact time naapo
model.eval()
start_time = time.time()
with torch.no_grad():
    win = [torch.sigmoid(model(single_ecg[:, :, s:s + WINDOW]))
           for s in range(0, single_ecg.shape[2] - WINDOW + 1, STRIDE)]
    probs = torch.stack(win).max(dim=0).values.cpu().numpy()[0]
end_time = time.time()

# 3. Result print karo!
classes = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
print(f'Inference Time: {(end_time - start_time)*1000:.2f} milliseconds\n')

print('Class | Model Probability | Predicted (0.5) | Actual Doctor Label')
print('-----------------------------------------------------------------')
for i, c in enumerate(classes):
  pred_bin = 1 if probs[i] >= thresholds[i] else 0
  print(
      f'{c:5s} |      {probs[i]*100:6.2f}%      |        {pred_bin}       '
      f' |          {int(true_label[i])}'
  )


Inference Time: 0.00 milliseconds

Class | Model Probability | Predicted (0.5) | Actual Doctor Label
-----------------------------------------------------------------
CD    |       11.05%      |        0        |          0
HYP   |        2.01%      |        0        |          0
MI    |       34.02%      |        0        |          1
NORM  |       82.14%      |        1        |          0
STTC  |        0.88%      |        0        |          0


Testing done , now we shall start chapman dataset 